In [ ]:
knitr::opts_chunk$set(echo = TRUE, warning = FALSE, message = FALSE, fig.align = 'center')
library(tidyverse)
library(imbalance)
library(GGally)
library(patchwork)
library(recipes)
library(themis)
library(smotefamily)
library(devtools)
library(formattable)
library(kableExtra)
library(pROC)
library(xgboost)
library(tidymodels)
library(xgboost)
library(torch)
library(unbalanced)
library(UBL)
library(caret)
library(conflicted)
conflict_prefer("filter", "dplyr")
conflict_prefer("train", "caret")
conflict_prefer("comma", "scales")
conflict_prefer("percent", "scales")

set.seed(456)

### 1. Data Preparation 💾

This dataset is used for predicting the likelihood of a **cerebral stroke** based on various health and demographic factors. It is an **imbalanced classification problem**, making it suitable for evaluating models using metrics like **Precision**, **Recall**, **F1 Score**, **ROC-AUC**, and **PR-AUC**.

**Key Features Include:**
- Age, Hypertension, Heart Disease
- Average Glucose Level, BMI
- Smoking Status, Gender, Work Type, Residence Type
- Stroke (target variable)

**Class Imbalance:** The target variable (`stroke`) is highly imbalanced, with a small percentage of positive stroke cases.

**Source:** [Cerebral Stroke Prediction (Imbalanced Dataset) on Kaggle](https://www.kaggle.com/datasets/shashwatwork/cerebral-stroke-predictionimbalaced-dataset)

In [ ]:

data_raw <- read.csv("data/dataset.csv")


factor_cols <- c("gender", "hypertension","heart_disease","ever_married",
                 "work_type","Residence_type","smoking_status")


data <- data_raw %>%
  mutate(Class = ifelse(stroke == 0, "negative","positive"))%>%
  select(-stroke)%>%
  mutate(Class = factor(Class, levels = c("negative","positive")))%>%
  select(-id)%>%
  mutate(across(all_of(factor_cols),as.factor))%>%
  ## Addressing NA
  mutate(smoking_status = ifelse(is.na(smoking_status), "Others",smoking_status))%>%
  filter(if_all(everything(), ~!is.na(.)))

### Find the best threshold based on the valuation criteria
best_threshold <- function(roc){
  
  roc_cords <- coords(roc,
                    ret = c("threshold", "ppv", "specificity", "sensitivity", "precision"),
                    transpose = FALSE)

  p <- roc_cords$precision
  s <- roc_cords$sensitivity
  roc_cords$f1 <- ifelse(p + s == 0, 0, 2 * p * s / (p + s))
  roc_cords <- roc_cords[!is.na(roc_cords$f1),]
  high_sens <- roc_cords
  # high_sens <- roc_cords [roc_cords['sensitivity']>= 0.9, ]
  best_thres <- high_sens[which.max(high_sens$f1), "threshold"]
  
  return(best_thres)
}
### Find the model + testing it to find the valuation criteria
evaluate_model <- function(model, valid_data, test_data) {
  
  is_torch_model <- inherits(model, "nn_module")
  is_xgb_model <- inherits(model, "xgb.Booster")

  predict_probs <- function(model, newdata) {
    if (is_torch_model) {
  x_mat <- torch_tensor(as.matrix(model.matrix(Class ~ . - 1, data = newdata)), 
                        dtype = torch_float())
  probs_valid <-as.numeric(as_array( model(x_mat)))
} else if (is_xgb_model) {
  x_mat <- model.matrix(Class ~ . - 1, data = newdata)
  probs_valid <- predict(model, x_mat)
} else {
  probs_valid <- predict(model, newdata = newdata, type = "prob")[, "positive"]
  
}
    
    return(probs_valid)
    }
  
  probs_valid <- predict_probs(model, valid_data)
  # ROC on validation
  roc_obj_valid <- roc(response = valid_data$Class,
                       predictor = probs_valid,
                       levels = c("negative", "positive"))
  
  # Find best threshold
  best_thresh <- best_threshold(roc_obj_valid)
  
  # Predict on test set
  probs_test <- predict_probs(model, test_data)
  
  # Convert probabilities to classes
  pred_classes <- ifelse(probs_test >= best_thresh, "positive", "negative")
  pred_classes <- factor(pred_classes, levels = c("negative", "positive"))
  
  # Confusion matrix
  cm_rf_smote_enn <- confusionMatrix(pred_classes, test_data$Class, positive = "positive")
  
  # Return results
  return(list(best_thresh = best_thresh, cm = cm_rf_smote_enn))
}
### Report the important valuation criteria that we care 
eval_fnc <- function(cm){

sensitivity <- cm$byClass["Sensitivity"]
specificity <- cm$byClass["Specificity"]
precision   <- cm$byClass["Pos Pred Value"]
f1_score    <- 2 * (precision * sensitivity) / (precision + sensitivity)

print(cm$table)

cat("\nSensitivity:", round(sensitivity, 3))
cat("\nSpecificity:", round(specificity, 3))
cat("\nPrecision:   ", round(precision, 3))
cat("\nF1 Score:   ", round(f1_score, 3), "\n")


metrics <- c(
  Sensitivity = cm$byClass["Sensitivity"],
  Specificity = cm$byClass["Specificity"],
  Precision   = cm$byClass["Pos Pred Value"],
  F1_Score    = cm$byClass["F1"]
)

return(metrics)


}

fontsize <- 8
basesize <- 8

___

### 2. Exploratory Data Analysis 🔍
#### 2.1. Initial Data Overview 📊

**Format**

A data frame with `r comma(nrow(data),0)` records and `r length(names(data))` columns; `r comma(nrow(data[data['Class'] == "positive",]),0)` comes with positive (minority) outcomes and `r comma(nrow(data[data['Class'] == "negative",]),0)` comes with negative (majority) outcomes (`r percent(nrow(data[data['Class'] == "positive",])/nrow(data),0)` imbalance). 

**Variables**

`r names(data)`

In [ ]:

glimpse(data)


___

#### 2.2. Visualization
###### 2.2.1. Distribution of Classes

In [ ]:

ggplot(data, aes(x = Class, fill = Class)) +
  geom_bar() +
  labs(title = "Unbalanced Distribution", x = "Class", y = "Count") +
  theme_minimal(base_size = basesize) +
  scale_fill_manual(values = c("positive" = "skyblue", "negative" = "tomato"))+
  theme(axis.title.x = element_text(size = fontsize),
        axis.text.x = element_text(size = fontsize),
        axis.title.y = element_text(size = fontsize),
        axis.text.y = element_text(size = fontsize),
        plot.title = element_text(size = fontsize),
        legend.title = element_text(size = fontsize),
        legend.text = element_text(size = fontsize),
        legend.key.size =  unit(2, 'mm'),
        panel.border = element_rect(color = "black", fill = NA, linewidth = 0.7))


___

###### 2.2.2. Distribution of Predictors

In [ ]:

numeric_predictors <- setdiff(names(data), c("Class",factor_cols))

plot_list <- list()

for (col_name in numeric_predictors) {
p <- ggplot(data, aes(x = .data[[col_name]])) +
    geom_histogram(aes(y = after_stat(density)), binwidth = NULL, fill = "lightblue", color = "black", alpha = 0.7) +
    geom_density(color = "darkblue", linewidth = 1) +
    labs(
      title = paste("Distribution of", col_name),
      x = col_name,
      y = "Density"
    ) +
    theme_minimal()
   plot_list[[col_name]] <- p
}
   

combined_plot <- wrap_plots(plot_list, ncol = 2) 

print(combined_plot)


___

###### 2.2.3. Distribution of Predictors by Class - Histogram

In [ ]:


plot_list <- list()
class_colors <- c("positive" = "skyblue", "negative" = "tomato")

for (col_name in numeric_predictors) {
  p <- ggplot(data, aes(x = .data[[col_name]], fill = Class, color = Class)) +
    geom_density(alpha = 0.5, linewidth = 0.8) +
    scale_fill_manual(values = class_colors) +
    scale_color_manual(values = class_colors) +
    labs(
      title = paste("Distribution of", col_name, "by Class"),
      x = col_name,
      y = "Density"
    ) +
    theme_minimal(base_size = fontsize -1) +
    theme(legend.position = "none")

  plot_list[[col_name]] <- p
}

combined_plot <- wrap_plots(plot_list, ncol = 2) +
                 plot_layout(guides = 'collect') +
                 theme(legend.position = 'right')

print(combined_plot)

###### 2.2.3. Distribution of Predictors by Class - Contingency Tables

In [ ]:

factor_predictors <- factor_cols  

tables_list <- list()

for (col_name in factor_predictors) {
  tbl_df <- data %>%
    group_by(!!sym(col_name), Class) %>%
    summarise(Count = n(), .groups = "drop") %>%
    group_by(!!sym(col_name)) %>%
    mutate(Percent = round(100 * Count / sum(Count), 1),
           Label = paste0(comma(Count), " (", Percent, "%)")) %>%
    select(!!sym(col_name), Class, Label) %>%
    pivot_wider(names_from = Class, values_from = Label) %>%
    mutate(across(everything(), ~replace_na(.x, "0 (0.0%)")))
  
  tables_list[[col_name]] <- kable(tbl_df, format = "html") %>%
    kable_styling(bootstrap_options = c("striped", "hover", "condensed", "responsive"),
                  full_width = FALSE, position = "left",
                  font_size = 12)
}


for (i in seq(1, length(tables_list), by = 4)) {
cat('<div style="display: flex; justify-content: flex-start; gap: 20px;">')

print(tables_list[[i]])

if ((i + 1) <= length(tables_list)) {
  print(tables_list[[i + 1]])
}

if ((i + 2) <= length(tables_list)) {
  print(tables_list[[i + 2]])
}

if ((i + 3) <= length(tables_list)) {
  print(tables_list[[i + 3]])
}

cat('</div><br>\n')
}

___

### 3. Methodology 🛠️

The following machine learning workflow is considered for all models:

**📦 Data Splitting:**  
The original dataset is divided into three parts: I used **stratified splits** to ensure the minority class is represented in all subsets. 

- **Training set (≈56%)**: used to train the model.  
- **Validation set (≈24%)**: used to tune hyperparameters and select the optimal classification threshold.  
- **Test set (≈20%)**: held out and used **only once** for final model evaluation.  

This 3-way split ensures that model selection and threshold tuning do not bias the final reported performance.

In [ ]:

test_index <- createDataPartition(data$Class, p = 0.8, list = FALSE)
data_train_valid <- data[test_index, ]
data_test <- data[-test_index, ]

train_index <- createDataPartition(data_train_valid$Class, p = 0.7, list = FALSE)
data_train <- data_train_valid[train_index, ]
data_valid <- data_train_valid[-train_index, ]


train_control <- trainControl(method = "cv",
                              number = 5,
                              classProbs = TRUE,
                              summaryFunction = twoClassSummary,
                              savePredictions = "final")



**⚖️ Handling Class Imbalance:**  
All training datasets are augmented using **oversampling techniques** such as **SMOTE**, **ADASYN**, or **SMOTE-ENN** to address the severe class imbalance (positive class ≈ 1%).  
However, both the **validation** and **test** sets are **not augmented** and retain their original class distributions to reflect real-world scenarios.

In addition to oversampling, we also apply **class weighting** during model training to further mitigate imbalance. The weighting scheme assigns higher penalties to misclassifications of the minority class. The weights are computed inversely proportional to class frequencies, using the formula:

\[
\text{weight}_{\text{class}} = \frac{n_{\text{total}}}{2 \times n_{\text{class}}}
\]

where \( n_{\text{class}} \) is the number of observations in a given class, and \( n_{\text{total}} \) is the total number of training samples.

**🎯 Model Training & Threshold Selection:**  
All models are trained to **maximize the F1 score**, which balances precision and recall, making it particularly suitable for highly imbalanced classification tasks. The classification **threshold** is tuned on the **validation set**. This tuned threshold is then applied to the test set for **final evaluation**, which is performed only once.

**📊 Evaluation Metrics:**  
Model performance is assessed using the following metrics:

- **F1 Score**: harmonic mean of precision and recall  
- **Precision (PPV)** and **Recall (Sensitivity)**  
- **Specificity** and **ROC-AUC** (where appropriate)  
- **PR-AUC** is also considered in some cases to better capture precision-recall trade-offs under imbalance  

This methodology ensures fair comparison and realistic performance estimation.

___

#### 3.1. Generalized Linear Models (GLM) 🔢

In this section, we evaluate **logistic regression (GLM)** for handling the imbalanced dataset.  
We explore the following strategies:

1. **Baseline logistic regression** without any class imbalance treatment.  
2. **Resampling approaches**: Random Over-Sampling (ROS), SMOTE, SMOTE-ENN, Random Under-Sampling (RUS)  

The classification **threshold is tuned on the validation set** to maximize the **F1 score**,  
ensuring that the model achieves a balance between **precision** and **recall** while accounting for the class imbalance.

##### 3.1.1 Baseline GLM

In [ ]:



glm_baseline <- train(Class ~ .,
                        data = data_train,
                        method = "glm",
                        trControl = train_control,
                        metric = "ROC")
                        ##"Recall", "Sensitivity", "Specificity"

glm_eval <- evaluate_model(glm_baseline, 
               data_valid,
               data_test)

cm_glm <- glm_eval$cm

best_thresh <- glm_eval$best_thresh 


ROC is used to find an optimal threshold for classification. This time, the threshold is `r round(best_thresh,2)`

In [ ]:

glm_eval <- eval_fnc(cm_glm)


##### 3.1.2 GLM + ROS

In [ ]:

data_train_ros <- upSample(
  x = subset(data_train, select = -Class),
  y = data_train$Class,
  yname = "Class")


table(data_train_ros$Class)

glm_ros <- train(Class ~ .,
                            data = data_train_ros,
                            method = "glm",
                            trControl = train_control,
                            metric = "ROC")

glm_ros_eval <- evaluate_model(glm_ros, 
               data_valid,
               data_test)

cm_glm_ros <- glm_ros_eval$cm

best_thresh <- glm_ros_eval$best_thresh 


ROC is used to find an optimal threshold for classification. This time, the threshold is `r round(best_thresh,2)`

In [ ]:

glm_ros_eval <- eval_fnc(cm_glm_ros)


___

##### 3.1.3 GLM + SMOTE

In [ ]:

prepped_recipe_smote <- recipe(Class ~ ., data = data_train) %>%
  step_dummy(all_nominal_predictors()) %>%
  step_smote(Class)%>%prep()

data_train_smote <- juice(prepped_recipe_smote)

data_valid_smote <- recipe(Class ~ ., data = data_valid) %>%
  step_dummy(all_nominal_predictors()) %>%
  prep() %>%
  bake(new_data = NULL)

data_test_smote <- recipe(Class ~ ., data = data_test) %>%
  step_dummy(all_nominal_predictors()) %>%
  prep() %>%
  bake(new_data = NULL)



glm_smote <- train(Class ~ .,
                              data = data_train_smote,
                              method = "glm",
                              family = "binomial",
                              trControl = train_control,
                              metric = "ROC")

glm_smote_eval <- evaluate_model(glm_smote, 
               data_valid_smote,
               data_test_smote)

cm_glm_smote <- glm_smote_eval$cm

best_thresh <- glm_smote_eval$best_thresh 


ROC is used to find an optimal threshold for classification. This time, the threshold is `r round(best_thresh,2)`

In [ ]:

glm_smote_eval <- eval_fnc(cm_glm_smote)


___

##### 3.1.4 GLM + SMOTE-ENN

In [ ]:

prepped_recipe_smote_enn <- recipe(Class ~ ., data = data_train) %>%
  step_dummy(all_nominal_predictors()) %>%
  step_smote(Class, over_ratio = 1)%>%prep()


data_train_smote_enn_start <- juice(prepped_recipe_smote_enn)
data_test_smote_enn <- recipe(Class ~ ., data = data_test) %>%
  step_dummy(all_nominal_predictors()) %>%
  prep() %>%
  bake(new_data = NULL)

data_valid_smote_enn <- recipe(Class ~ ., data = data_valid) %>%
  step_dummy(all_nominal_predictors()) %>%
  prep() %>%
  bake(new_data = NULL)

data_train_smote_enn_start$Class <- ifelse(
  data_train_smote_enn_start$Class == "negative", 0, 1)

data_train_smote_enn_start <- ubENN(X = data_train_smote_enn_start
                    [, setdiff(names(data_train_smote_enn_start), "Class")],
                    Y =  data_train_smote_enn_start$Class)

data_train_smote_enn <-
  cbind(data_train_smote_enn_start$X,
        data.frame(Class = factor(
          ifelse(
          data_train_smote_enn_start$Y == 0, 'negative','positive'),
          level = c('negative','positive'))))

table(data_train_smote_enn$Class)

glm_smote_enn <- train(Class ~ .,
                              data = data_train_smote_enn,
                              method = "glm",
                              family = "binomial",
                              trControl = train_control,
                              metric = "ROC")

glm_smote_enn_eval <- evaluate_model(glm_smote_enn, 
               data_valid_smote_enn,
               data_test_smote_enn)

cm_glm_smote_enn <- glm_smote_enn_eval$cm

best_thresh <- glm_smote_enn_eval$best_thresh 


ROC is used to find an optimal threshold for classification. This time, the threshold is `r round(best_thresh,2)`

In [ ]:

glm_smote_enn_eval <- eval_fnc(cm_glm_smote_enn)


___

##### 3.1.5 GLM + RUS 

In [ ]:

data_train_rus <- downSample(
  x = subset(data_train, select = -Class),
  y = data_train$Class,
  yname = "Class")

table(data_train_rus$Class)

glm_rus <- train(Class ~ .,
                            data = data_train_rus,
                            method = "glm",
                            trControl = train_control,
                            metric = "ROC")

glm_rus_eval <- evaluate_model(glm_rus, 
               data_valid,
               data_test)

cm_rus <- glm_rus_eval$cm

best_thresh <- glm_rus_eval$best_thresh 


ROC is used to find an optimal threshold for classification. This time, the threshold is `r round(best_thresh,2)`

In [ ]:

glm_rus_eval <- eval_fnc(cm_rus)


___

#### 3.2 Random Forest 🌲

Random Forest is a robust ensemble method that often performs well with unbalanced data due to its ability to handle non-linear relationships.  
In this section, we test different **Random Forest variations**:

1. **Baseline logistic regression** without any class imbalance treatment.  
2. **Resampling approaches**: SMOTE-ENN and ADASYN
3. **Cost-Sensitive Weights**

The classification **threshold is tuned on the validation set** to maximize the **F1 score**,  
ensuring that the model achieves a balance between **precision** and **recall** while accounting for the class imbalance.

##### 3.2.1. Baseline RF

In [ ]:


#  Random Forest
rf <- train(Class ~ .,
                     data = data_train,
                     method = "ranger",
                     trControl = train_control,
                     metric = "ROC",
                     num.trees = 100,
                     importance = 'permutation'
                    )

rf_eval <- evaluate_model(rf, 
               data_valid,
               data_test)

cm_rf <- rf_eval$cm

best_thresh <- rf_eval$best_thresh 


ROC is used to find an optimal threshold for classification. This time, the threshold is `r round(best_thresh,2)`

In [ ]:

rf_eval <- eval_fnc(cm_rf)


___

##### 3.2.2 RF + SMOTE

In [ ]:

rf_smote <- train(Class ~ .,
                     data = data_train_smote,
                     method = "ranger",
                     trControl = train_control,
                     metric = "ROC",
                     num.trees = 100, ## Default is 500
                     importance = 'permutation'
                    )

rf_smote_eval <- evaluate_model(rf_smote, 
               data_valid_smote,
               data_test_smote)

cm_rf_smote <- rf_smote_eval$cm

best_thresh <- rf_smote_eval$best_thresh 


ROC is used to find an optimal threshold for classification. This time, the threshold is `r round(best_thresh,2)`

In [ ]:

rf_smote_eval <- eval_fnc(cm_rf_smote)


___

##### 3.2.3 RF + SMOTE-ENN

In [ ]:

rf_smote_enn <- train(Class ~ .,
                     data = data_train_smote_enn,
                     method = "ranger",
                     trControl = train_control,
                     metric = "ROC",
                     num.trees = 100, ## Default is 500
                     importance = 'permutation'
                    )

rf_smote_enn_eval <- evaluate_model(rf_smote_enn, 
               data_valid_smote_enn,
               data_test_smote_enn)

cm_rf_smote_enn <- rf_smote_enn_eval$cm

best_thresh <- rf_smote_enn_eval$best_thresh 


ROC is used to find an optimal threshold for classification. This time, the threshold is `r round(best_thresh,2)`

In [ ]:

rf_smote_enn_eval <- eval_fnc(cm_rf_smote_enn)


___

##### 3.2.4 RF + ADASYN

In [ ]:

data_train$Class <- as.factor(data_train$Class)

X_train <- model.matrix(Class ~ . - 1, data = data_train) 
X_train <- as.data.frame(X_train)
X_train$Class <- data_train$Class
## The reason for using model.matrix is to ensure all factors are well transformed 
## into integer numbers

data_train_adasyn <- AdasynClassif(Class ~ ., X_train)


rf_adasyn <- train(Class ~ .,
                     data = data_train_adasyn,
                     method = "ranger",
                     trControl = train_control,
                     metric = "ROC",
                     num.trees = 100, ## Default is 500
                     importance = 'permutation'
                    )

X_valid <- model.matrix(Class ~ . - 1, data = data_valid)
X_valid <- as.data.frame(X_valid)
X_valid$Class <- data_valid$Class


X_test <- model.matrix(Class ~ . - 1, data = data_test)
X_test <- as.data.frame(X_test)
X_test$Class <- data_test$Class

rf_adasyn_eval <- evaluate_model(rf_adasyn, 
               X_valid,
               X_test)

cm_rf_adasyn <- rf_adasyn_eval$cm

best_thresh <- rf_adasyn_eval$best_thresh 


ROC is used to find an optimal threshold for classification. This time, the threshold is `r round(best_thresh,2)`

In [ ]:

rf_adasyn_eval <- eval_fnc(cm_rf_adasyn)


##### 3.2.5 Weighted RF

In [ ]:


n_negative_train <- sum(data_train$Class == "negative")
n_positive_train <- sum(data_train$Class == "positive")

model_weights <- c(
  negative = (n_negative_train + n_positive_train) / (2 * n_negative_train),
  positive = (n_negative_train + n_positive_train) / (2 * n_positive_train))


# Weighted Random Forest
weighted_rf_default <- train(Class ~ .,
                     data = data_train,
                     method = "ranger",
                     trControl = train_control,
                     metric = "ROC",
                     class.weights = model_weights,
                     num.trees = 100, ## Default is 500
                     importance = 'permutation'
                    )

weighted_rf_eval <- evaluate_model(weighted_rf_default, 
               data_valid,
               data_test)

cm_weighted_rf <- weighted_rf_eval$cm

best_thresh <- weighted_rf_eval$best_thresh 


ROC is used to find an optimal threshold for classification. This time, the threshold is `r round(best_thresh,2)`

In [ ]:

rf_weighted_eval <- eval_fnc(cm_weighted_rf)


___

#### 3.3 Boosting (XGBoost) ⚡

In this section, we explore the **XGBoost** algorithm, a powerful gradient boosting technique, for handling the severe class imbalance.

We compare the following strategies:
1. **Baseline logistic regression** without any class imbalance treatment. 
2. **Resampling approaches**: **SMOTE**, **SMOTE-ENN**, and **ADASYN**.  
3. **Cost-Sensitive Weights**

As with previous models, the **classification threshold** is tuned on the **validation set** to maximize the **F1 score**, ensuring a balance between precision and recall.

##### 3.3.1 Baseline XGBoost 

In [ ]:

x_train <- model.matrix(Class ~ . - 1, data = data_train)
y_train <- ifelse(data_train$Class == "positive", 1, 0)

xgb <- xgboost(
  data = x_train,
  label = y_train,
  objective = "binary:logistic",
  nrounds = 100,
  eval_metric = "auc",
  verbose = 0  # Suppress output
)

xgb_eval <- evaluate_model(xgb, 
               data_valid,
               data_test)

cm_xgb <- xgb_eval$cm

best_thresh <- xgb_eval$best_thresh 


ROC is used to find an optimal threshold for classification. This time, the threshold is `r round(best_thresh,2)`

In [ ]:

xgb_eval <- eval_fnc(cm_xgb)


___


##### 3.3.2 XGBoost + SMOTE 

In [ ]:

x_train_smote <- model.matrix(Class ~ . - 1, data = data_train_smote)
y_train_smote <- ifelse(data_train_smote$Class == "positive", 1, 0)

xgb_smote <- xgboost(
  data = x_train_smote,
  label = y_train_smote,
  objective = "binary:logistic",
  nrounds = 100,
  eval_metric = "auc",
  verbose = 0  # Suppress output
)

xgb_smote_eval <- evaluate_model(xgb_smote, 
               data_valid_smote,
               data_test_smote)

cm_xgb_smote <- xgb_smote_eval$cm

best_thresh <- xgb_smote_eval$best_thresh 



Again, ROC is used to find an optimal threshold for classification. This time, the threshold is `r round(best_thresh,2)`

In [ ]:

xgb_smote_eval <- eval_fnc(cm_xgb_smote)


##### 3.3.3 XGBoost + SMOTE-ENN

In [ ]:

x_train_smote_enn <- model.matrix(Class ~ . - 1, data = data_train_smote_enn)
y_train_smote_enn <- ifelse(data_train_smote_enn$Class == "positive", 1, 0)

xgb_smote_enn <- xgboost(
  data = x_train_smote_enn,
  label = y_train_smote_enn,
  objective = "binary:logistic",
  nrounds = 100,
  eval_metric = "auc",
  verbose = 0  # Suppress output
)

xgb_smote_enn_eval <- evaluate_model(xgb_smote_enn, 
               data_valid_smote_enn,
               data_test_smote_enn)

cm_xgb_smote_enn <- xgb_smote_enn_eval$cm

best_thresh <- xgb_smote_enn_eval$best_thresh 



Again, ROC is used to find an optimal threshold for classification. This time, the threshold is `r round(best_thresh,2)`

In [ ]:

xgb_smote_enn_eval <- eval_fnc(cm_xgb_smote_enn)


##### 3.3.4 XGBoost + ADASYN


In [ ]:

x_train_adasyn <- model.matrix(Class ~ . - 1, data = data_train_adasyn)
y_train_adasyn <- ifelse(data_train_adasyn$Class == "positive", 1, 0)


xgb_adasyn <- xgboost(
  data = x_train_adasyn,
  label = y_train_adasyn,
  objective = "binary:logistic",
  nrounds = 100,
  eval_metric = "auc",
  verbose = 0  # Suppress output
)

xgb_adasyn_eval <- evaluate_model(xgb_adasyn, 
               X_valid,
               X_test)

cm_xgb_adasyn <- xgb_adasyn_eval$cm

best_thresh <- xgb_adasyn_eval$best_thresh 



Again, ROC is used to find an optimal threshold for classification. This time, the threshold is `r round(best_thresh,2)`

In [ ]:

xgb_adasyn_eval <- eval_fnc(cm_xgb_adasyn)


##### 3.3.5 Weighted XGBoost

In [ ]:


weight_negative <- (n_negative_train + n_positive_train) / (2 * n_negative_train)
weight_positive <- (n_negative_train + n_positive_train) / (2 * n_positive_train)

# Create a vector of weights per row
row_weights <- ifelse(y_train == 1, model_weights[['positive']], model_weights[['negative']])


weighted_xgb <- xgboost(
  data = x_train,
  label = y_train,
  objective = "binary:logistic",
  nrounds = 100,
  eval_metric = "auc",
  weight = row_weights,
  verbose = 0  # Suppress output
)

weighted_xgb_eval <- evaluate_model(weighted_xgb, 
               data_valid,
               data_test)

cm_weighted_xgb <- weighted_xgb_eval$cm

best_thresh <- weighted_xgb_eval$best_thresh 


ROC is used to find an optimal threshold for classification. This time, the threshold is `r round(best_thresh,2)`

In [ ]:

xgb_weighted_eval <- eval_fnc(cm_weighted_xgb)


#### 3.4  Neural Networks (NN) 🧠

In this section, we explore **feed-forward Neural Networks (NN)** for the same task. Neural networks can capture complex patterns but are also sensitive to class imbalance.

We compare the following strategies:
1. **Baseline logistic regression** without any class imbalance treatment. 
2. **Resampling approaches**: **SMOTE**, **SMOTE-ENN**, and **ADASYN**.  
3. **Cost-Sensitive Weights**

As with previous models, the **classification threshold** is tuned on the **validation set** to maximize the **F1 score**, ensuring a balance between precision and recall.

##### 3.4.1 Baseline NN 

In [ ]:

x_train <- torch_tensor(as.matrix(model.matrix(Class ~ . - 1, data = data_train)), 
                        dtype = torch_float())
y_train <- torch_tensor(ifelse(data_train$Class == "positive", 1, 0), 
                        dtype = torch_float())$unsqueeze(2)

nn_define <- nn_module(
  "Net",
  initialize = function(input_dim) {
    self$fc1 <- nn_linear(input_dim, 16)
    self$dropout <- nn_dropout(p = 0.3)
    self$fc2 <- nn_linear(16, 8)
    self$fc3 <- nn_linear(8, 1)
  },
  forward = function(x) {
    x %>%
      self$fc1() %>%
      nnf_relu() %>%
      self$dropout() %>%
      self$fc2() %>%
      nnf_relu() %>%
      self$fc3() %>%
      nnf_sigmoid()
  }
)

nn <- nn_define(input_dim = ncol(x_train))

optimizer <- optim_adam(nn$parameters, lr = 0.001)
loss_fn <- nn_bce_loss()

for (epoch in 1:100) {
  nn$train()
  optimizer$zero_grad()
  output <- nn(x_train)
  loss <- loss_fn(output, y_train)
  loss$backward()
  optimizer$step()

  if (epoch %% 10 == 0) {
    cat(sprintf("Epoch %d, Loss: %.4f\n", epoch, loss$item()))
  }
}

nn_eval <- evaluate_model(nn, 
               data_valid,
               data_test)

cm_nn <- nn_eval$cm

best_thresh <- nn_eval$best_thresh 


ROC is used to find an optimal threshold for classification. This time, the threshold is `r round(best_thresh,2)`

In [ ]:

nn_eval <- eval_fnc(cm_nn)


___

##### 3.3.2 NN + SMOTE 

In [ ]:

x_train_smote <- torch_tensor(as.matrix(model.matrix(Class ~ . - 1, data = data_train_smote)), 
                        dtype = torch_float())
y_train_smote <- torch_tensor(ifelse(data_train_smote$Class == "positive", 1, 0), 
                        dtype = torch_float())$unsqueeze(2)

nn_smote <- nn_define(input_dim = ncol(x_train_smote))

optimizer <- optim_adam(nn_smote$parameters, lr = 0.001)
loss_fn <- nn_bce_loss()

for (epoch in 1:100) {
  nn_smote$train()
  optimizer$zero_grad()
  output <- nn_smote(x_train_smote)
  loss <- loss_fn(output, y_train_smote)
  loss$backward()
  optimizer$step()

  if (epoch %% 10 == 0) {
    cat(sprintf("Epoch %d, Loss: %.4f\n", epoch, loss$item()))
  }
}

nn_smote_eval <- evaluate_model(nn_smote, 
               data_valid_smote,
               data_test_smote)

cm_nn_smote <- nn_smote_eval$cm

best_thresh <- nn_smote_eval$best_thresh 


ROC is used to find an optimal threshold for classification. This time, the threshold is `r round(best_thresh,2)`

In [ ]:

nn_smote_eval <- eval_fnc(cm_nn_smote)


##### 3.3.2 NN + SMOTE-ENN 

In [ ]:

x_train_smote_enn <- torch_tensor(as.matrix(model.matrix(Class ~ . - 1, data = data_train_smote_enn)), 
                        dtype = torch_float())
y_train_smote_enn <- torch_tensor(ifelse(data_train_smote_enn$Class == "positive", 1, 0), 
                        dtype = torch_float())$unsqueeze(2)

nn_smote_enn <- nn_define(input_dim = ncol(x_train_smote_enn))

optimizer <- optim_adam(nn_smote_enn$parameters, lr = 0.001)
loss_fn <- nn_bce_loss()

for (epoch in 1:100) {
  nn_smote_enn$train()
  optimizer$zero_grad()
  output <- nn_smote_enn(x_train_smote)
  loss <- loss_fn(output, y_train_smote)
  loss$backward()
  optimizer$step()

  if (epoch %% 10 == 0) {
    cat(sprintf("Epoch %d, Loss: %.4f\n", epoch, loss$item()))
  }
}

nn_smote_enn_eval <- evaluate_model(nn_smote_enn, 
               data_valid_smote_enn,
               data_test_smote_enn)

cm_nn_smote_enn <- nn_smote_enn_eval$cm

best_thresh <- nn_smote_enn_eval$best_thresh 


ROC is used to find an optimal threshold for classification. This time, the threshold is `r round(best_thresh,2)`

In [ ]:

nn_smote_enn_eval <- eval_fnc(cm_nn_smote_enn)


##### 3.3.4 NN + ADASYN


In [ ]:

x_train_adasyn <- torch_tensor(
  as.matrix(model.matrix(Class ~ . - 1, data = data_train_adasyn)), 
                        dtype = torch_float())
y_train_adasyn <- torch_tensor(
  ifelse(data_train_adasyn$Class == "positive", 1, 0), 
                        dtype = torch_float())$unsqueeze(2)

nn_adasyn <- nn_define(input_dim = ncol(x_train_adasyn))

optimizer <- optim_adam(nn_smote_enn$parameters, lr = 0.001)
loss_fn <- nn_bce_loss()

for (epoch in 1:100) {
  nn_adasyn$train()
  optimizer$zero_grad()
  output <- nn_adasyn(x_train_adasyn)
  loss <- loss_fn(output, y_train_adasyn)
  loss$backward()
  optimizer$step()

  if (epoch %% 10 == 0) {
    cat(sprintf("Epoch %d, Loss: %.4f\n", epoch, loss$item()))
  }
}


nn_adasyn_eval <- evaluate_model(nn_adasyn, 
               X_valid,
               X_test)

cm_nn_adasyn <- nn_adasyn_eval$cm

best_thresh <- nn_adasyn_eval$best_thresh 



Again, ROC is used to find an optimal threshold for classification. This time, the threshold is `r round(best_thresh,2)`

In [ ]:

nn_adasyn_eval <- eval_fnc(cm_nn_adasyn)


##### 3.3.5 Weighted NN

In [ ]:

# Create a vector of weights per row
NN_weights_start <- list(
  negative = weight_negative,
  positive = weight_positive
)

nn_weighted <- nn_define(input_dim = ncol(x_train))


NN_weights <- torch_where(y_train == 1,
                           torch_tensor(NN_weights_start$positive),
                           torch_tensor(NN_weights_start$negative))

loss_fn <- function(output, target, weights) {
  nnf_binary_cross_entropy(output, target, weight = weights)
}

for (epoch in 1:100) {
  nn_weighted$train()
  optimizer$zero_grad()
  output <- nn_weighted(x_train)
  loss <- loss_fn(output, y_train, NN_weights)
  loss$backward()
  optimizer$step()

  if (epoch %% 10 == 0) {
    cat(sprintf("Epoch %d, Loss: %.4f\n", epoch, loss$item()))
  }
}


weighted_nn_eval <- evaluate_model(nn_weighted, 
               data_valid,
               data_test)

cm_weighted_nn <- weighted_nn_eval$cm

best_thresh <- weighted_nn_eval$best_thresh 


ROC is used to find an optimal threshold for classification. This time, the threshold is `r round(best_thresh,2)`

In [ ]:

nn_weighted_eval <- eval_fnc(cm_weighted_nn)


### 3.5 Model Comparison 📊

In this section, comparison of all four models—**GLM**, **Random Forest**, **XGBoost**, and **Neural Networks**—under six different strategies for handling class imbalance:

- **Baseline** (no adjustment)  
- **Random Oversampling (ROS)**  
- **SMOTE**  
- **SMOTE-ENN**  
- **Adaptive Synthetic (ADASYN)**  
- **Class-Sensitive Weighting**

For each combination, we report the **F1 Score**, **Precision**, **Recall**, and **AUC**, evaluated on the test set. Thresholds were tuned on the validation set to maximize **F1 Score**, unless otherwise noted.

In [ ]:

results_list <- list(
  "GLM (Baseline)"     = glm_eval,
  "GLM (SMOTE)"        = glm_smote_eval,
  "GLM (SMOTE-ENN)"    = glm_smote_enn_eval,
  
  "RF (Baseline)"      = rf_eval,
  "RF (SMOTE)"         = rf_smote_eval,
  "RF (SMOTE-ENN)"     = rf_smote_enn_eval,
  "RF (ADASYN)"        = rf_adasyn_eval,
  "RF (Weighting)"     = rf_weighted_eval,

  "XGBoost (Baseline)" = xgb_eval,
  "XGBoost (SMOTE)"    = xgb_smote_eval,
  "XGBoost (SMOTE-ENN)"= xgb_smote_enn_eval,
  "XGBoost (ADASYN)"   = xgb_adasyn_eval,
  "XGBoost (Weighting)"= xgb_weighted_eval,
  
  "NN (Baseline)"      = nn_eval,
  "NN (SMOTE)"         = nn_smote_eval ,
  "NN (SMOTE-ENN)"     = nn_smote_enn_eval,
  "NN (ADASYN)"        = nn_adasyn_eval,
  "NN (Weighting)"     = nn_weighted_eval
)

comparison_table <- data.frame(do.call(rbind, results_list))

comparison_table <- comparison_table %>%
  rename(
    Sensitivity = `Sensitivity.Sensitivity`,
    Specificity = `Specificity.Specificity`,
    Precision = `Precision.Pos.Pred.Value`,
    F1_Score = `F1_Score.F1`
  )
comparison_table <- comparison_table %>%
  arrange(desc(F1_Score))

# Step 2: Identify the maximum F1 value
max_f1 <- round(quantile(comparison_table$F1_Score, probs = 0.75, na.rm = TRUE), 3)
max_sensitivity <- round(quantile(comparison_table$Sensitivity, probs = 0.75,na.rm = TRUE), 3)
max_specifity<- round(quantile(comparison_table$Specificity, probs = 0.75,na.rm = TRUE), 3)
max_precision<- round(quantile(comparison_table$Precision, probs = 0.75,na.rm = TRUE), 3)


# 3. Display the comparison table using kable for a clean format 📊
comparison_table %>%
  mutate(
    F1_Score = round(F1_Score, 3),  # Round first
    F1_Score = ifelse(
      F1_Score >= max_f1,
      cell_spec(F1_Score, background = "#90EE90", bold = TRUE),  # Light green highlight
      as.character(F1_Score)
    )
  ) %>%
    mutate(
    Sensitivity = round(Sensitivity, 3),  # Round first
    Sensitivity = ifelse(
      Sensitivity >= max_sensitivity,
      cell_spec(Sensitivity, background = "#90EE90", bold = TRUE),  # Light green highlight
      as.character(Sensitivity)
    )
  ) %>%
    mutate(
    Specificity = round(Specificity, 3),  # Round first
    Specificity = ifelse(
      Specificity >= max_specifity,
      cell_spec(Specificity, background = "#90EE90", bold = TRUE),  # Light green highlight
      as.character(Specificity)
    )
  ) %>%
  mutate(
    Precision = round(Precision, 3),  # Round first
    Precision = ifelse(
      Precision >= max_precision,
      cell_spec(Precision, background = "#90EE90", bold = TRUE),  # Light green highlight
      as.character(Precision)
    )
  ) %>%
  knitr::kable(
    caption = "Model Performance Comparison",
    format = "html",
    escape = FALSE,  # Required for cell_spec HTML rendering
    digits = 3
  ) %>%
  kable_styling(
    bootstrap_options = c("striped", "hover", "condensed", "responsive"),
    full_width = FALSE,
    position = "center"
  ) %>%
  row_spec(0, bold = TRUE, background = "#D3D3D3") %>%
  footnote(
    general = "Performance metrics for various classification models- those above 75% quantile are highlighted green"
  )

### 3.6 Improving the Best Model 🚀

Section 3.5 showed that the Random Forest model with class weighting achieved the best performance compared to other models. Building on that, I am experimenting with different weighting schemes to see if further improvements are possible.

So far, I have been using the following weighting scheme:

\[
\text{weight}_{\text{class}} = \frac{n_{\text{total}}}{2 \times n_{\text{class}}}
\]

where:  
- \( n_{\text{class}} \) is the number of samples in the given class  
- \( n_{\text{total}} \) is the total number of samples in the training data

Now, I am testing alternative weighting strategies such as:

1. Adjusting the denominator to values other than 2
\[
\text{weight}_{\text{class}} = \frac{n_{\text{total}}}{r \times n_{\text{class}}}
\]

2. Applying a direct inverse frequency weighting:  
  \[
  \text{weight}_{\text{class}} = \frac{1}{n_{\text{class}}}
  \]
3. Applying Weighted Cross-Entropy (WCE)

  \[
  \text{weight}_{\text{major}} = w \cdot \log(p_{major})
  \text{weight}_{\text{minor}} = \log(1 - p_{major})
  \]

- \( w \): Class weight applied to the positive class to increase its importance
- \( p \): Model-predicted probability of the positive class

4. Applying Focal Loss 
  \[
  \text{weight}_{\text{major}} = (1-p)^{\gamma} \log(p)
  \text{weight}_{\text{minor}} = p^{\gamma} \log(1-p)
  \]

Additionally, I am combining these weighting schemes with advanced resampling techniques such as **SMOTE** (Synthetic Minority Over-sampling Technique) and **SMOTE-ENN** (a hybrid of SMOTE and Edited Nearest Neighbors) to further balance the dataset and improve model generalization.

These combined approaches aim to increase the model’s ability to correctly classify the minority class without sacrificing overall accuracy.

##### 3.6.1 Adjusting Weighting Scale (r)
\[
\text{weight}_{\text{class}} = \frac{n_{\text{total}}}{r \times n_{\text{class}}}
\]

In [ ]:

ratio_to_try <- c(0.5, 1.0, 1.5, 2.0, 3.0) # Example ratios relative to the inverse frequency

for (r in ratio_to_try) {
  model_weights <- c(
    negative = (n_negative_train + n_positive_train) / (r * n_negative_train),
    positive =  (n_negative_train + n_positive_train) / (r * n_positive_train)
  )

weighted_rf <- train(Class ~ .,
                     data = data_train,
                     method = "ranger",
                     trControl = train_control,
                     metric = "ROC",
                     class.weights = model_weights,
                     num.trees = 100, ## Default is 500
                     importance = 'permutation'
                    )

weighted_rf_eval <- evaluate_model(weighted_rf, 
               data_valid,
               data_test)

cm_weighted_rf <- weighted_rf_eval$cm

best_thresh <- weighted_rf_eval$best_thresh 

rf_weighted_eval <- eval_fnc(cm_weighted_rf)

var_name <- paste0("rf_weighted_eval_ratio_", r)
assign(var_name, rf_weighted_eval)
}



##### 3.6.2 Direct Inverse Frequency Weighting
  \[
  \text{weight}_{\text{class}} = \frac{1}{n_{\text{class}}}
  \]

In [ ]:


model_weights_inverse <- c(
  negative = (1 / n_negative_train) ,
  positive = (1 / n_positive_train)
)

weighted_rf_inverse <- train(Class ~ .,
                     data = data_train,
                     method = "ranger",
                     trControl = train_control,
                     metric = "ROC",
                     class.weights = model_weights_inverse,
                     num.trees = 100, ## Default is 500
                     importance = 'permutation'
                    )

weighted_rf_eval_inverse <- evaluate_model(weighted_rf_inverse, 
               data_valid,
               data_test)

cm_weighted_rf_inverse <- weighted_rf_eval_inverse$cm

rf_weighted_eval_weight_inverse <- eval_fnc(cm_weighted_rf_inverse)


##### 3.6.3  Weighted Cross-Entropy (WCE) with Different Ratios

  \[
  \text{weight}_{\text{major}} = - w \cdot \log(p_{major}) \\
  \text{weight}_{\text{minor}} = - \log(1 - p_{major})
  \]

In [ ]:

n_total_train <- n_negative_train + n_positive_train

p_positive <- n_positive_train / n_total_train
p_negative <- n_negative_train / n_total_train


for (w in c(0.5, 1, 1.5, 2, 3, 4)) {

model_weights_WCE <- c(
  negative = - w * log(p_negative),
  positive = - log(1 - p_negative)  # log(1 - p_major)
)

weighted_rf_WCE <- train(Class ~ .,
                     data = data_train,
                     method = "ranger",
                     trControl = train_control,
                     metric = "ROC",
                     class.weights = model_weights_WCE,
                     num.trees = 100, ## Default is 500
                     importance = 'permutation'
                    )


weighted_rf_eval_WCE <- evaluate_model(weighted_rf_WCE, 
               data_valid,
               data_test)




cm_weighted_rf_WCE <- weighted_rf_eval_WCE$cm

rf_weighted_eval <- eval_fnc(cm_weighted_rf_WCE)

var_name <- paste0("rf_weighted_eval_ratio_WCE_", w)
assign(var_name, rf_weighted_eval)
}






##### 3.6.4  Focal Loss with Different Ratios

  \[
  \text{weight}_{\text{major}} = (1-p)^{\gamma} \log(p) \\
  \text{weight}_{\text{minor}} = p^{\gamma} \log(1-p)
  \]
  

In [ ]:


gamma <- c(0.5, 1.0, 1.5, 2.0, 3.0, 4.0) # Example ratios relative to the inverse frequency

for (g in gamma) {
w_major <- (1 - p_negative)^g * log(p_negative)
w_minor <- (p_negative)^g * log(1 - p_negative)

model_weights <- c(
  negative = abs(w_major),
  positive = abs(w_minor)
)


weighted_rf <- train(Class ~ .,
                     data = data_train,
                     method = "ranger",
                     trControl = train_control,
                     metric = "ROC",
                     class.weights = model_weights,
                     num.trees = 100, ## Default is 500
                     importance = 'permutation'
                    )

weighted_rf_eval <- evaluate_model(weighted_rf, 
               data_valid,
               data_test)


cm_weighted_rf <- weighted_rf_eval$cm

best_thresh <- weighted_rf_eval$best_thresh 

rf_weighted_eval <- eval_fnc(cm_weighted_rf)

var_name <- paste0("rf_weighted_eval_ratio_FL_", g)
assign(var_name, rf_weighted_eval)
}





##### 3.6.5 Comparing Weighting Schems

In [ ]:

rf_weighted_eval_df <- data.frame(
  Weighting = c("r = 0.5", "r = 1", "r = 1.5", "r = 2", "r = 3", 
                "Inverse Frequency",
                "WCE - Weights = 0.5", "WCE - Weights = 1", "WCE - Weights = 1.5",
                "WCE - Weights = 2", "WCE - Weights = 3", "WCE - Weights = 4",
                "FL - Gamma = 0.5", "FL - Gamma = 1", "FL - Gammma = 1.5",
                "FL - Gamma = 2", "FL - Gamma = 3", "FL - Gamma = 4"),
  Sensitivity = c(
    rf_weighted_eval_ratio_0.5["Sensitivity.Sensitivity"],
    rf_weighted_eval_ratio_1["Sensitivity.Sensitivity"],
    rf_weighted_eval_ratio_1.5["Sensitivity.Sensitivity"],
    rf_weighted_eval_ratio_2["Sensitivity.Sensitivity"],
    rf_weighted_eval_ratio_3["Sensitivity.Sensitivity"],
    rf_weighted_eval_weight_inverse["Sensitivity.Sensitivity"],
    rf_weighted_eval_ratio_WCE_0.5["Sensitivity.Sensitivity"],
    rf_weighted_eval_ratio_WCE_1["Sensitivity.Sensitivity"],
    rf_weighted_eval_ratio_WCE_1.5["Sensitivity.Sensitivity"],
    rf_weighted_eval_ratio_WCE_2["Sensitivity.Sensitivity"],
    rf_weighted_eval_ratio_WCE_3["Sensitivity.Sensitivity"],
    rf_weighted_eval_ratio_WCE_4["Sensitivity.Sensitivity"],
    rf_weighted_eval_ratio_FL_0.5["Sensitivity.Sensitivity"],
    rf_weighted_eval_ratio_FL_1["Sensitivity.Sensitivity"],
    rf_weighted_eval_ratio_FL_1.5["Sensitivity.Sensitivity"],
    rf_weighted_eval_ratio_FL_2["Sensitivity.Sensitivity"],
    rf_weighted_eval_ratio_FL_3["Sensitivity.Sensitivity"],
    rf_weighted_eval_ratio_FL_4["Sensitivity.Sensitivity"]

  ),
  Specificity = c(
    rf_weighted_eval_ratio_0.5["Specificity.Specificity"],
    rf_weighted_eval_ratio_1["Specificity.Specificity"],
    rf_weighted_eval_ratio_1.5["Specificity.Specificity"],
    rf_weighted_eval_ratio_2["Specificity.Specificity"],
    rf_weighted_eval_ratio_3["Specificity.Specificity"],
    rf_weighted_eval_weight_inverse["Specificity.Specificity"],
    rf_weighted_eval_ratio_WCE_0.5["Specificity.Specificity"],
    rf_weighted_eval_ratio_WCE_1["Specificity.Specificity"],
    rf_weighted_eval_ratio_WCE_1.5["Specificity.Specificity"],
    rf_weighted_eval_ratio_WCE_2["Specificity.Specificity"],
    rf_weighted_eval_ratio_WCE_3["Specificity.Specificity"],
    rf_weighted_eval_ratio_WCE_4["Specificity.Specificity"],
    rf_weighted_eval_ratio_FL_0.5["Specificity.Specificity"],
    rf_weighted_eval_ratio_FL_1["Specificity.Specificity"],
    rf_weighted_eval_ratio_FL_1.5["Specificity.Specificity"],
    rf_weighted_eval_ratio_FL_2["Specificity.Specificity"],
    rf_weighted_eval_ratio_FL_3["Specificity.Specificity"],
    rf_weighted_eval_ratio_FL_4["Specificity.Specificity"]
  ),
  Precision = c(
    rf_weighted_eval_ratio_0.5["Precision.Pos Pred Value"],
    rf_weighted_eval_ratio_1["Precision.Pos Pred Value"],
    rf_weighted_eval_ratio_1.5["Precision.Pos Pred Value"],
    rf_weighted_eval_ratio_2["Precision.Pos Pred Value"],
    rf_weighted_eval_ratio_3["Precision.Pos Pred Value"],
    rf_weighted_eval_weight_inverse["Precision.Pos Pred Value"],
    rf_weighted_eval_ratio_WCE_0.5["Precision.Pos Pred Value"],
    rf_weighted_eval_ratio_WCE_1["Precision.Pos Pred Value"],
    rf_weighted_eval_ratio_WCE_1.5["Precision.Pos Pred Value"],
    rf_weighted_eval_ratio_WCE_2["Precision.Pos Pred Value"],
    rf_weighted_eval_ratio_WCE_3["Precision.Pos Pred Value"],
    rf_weighted_eval_ratio_WCE_4["Precision.Pos Pred Value"],
    rf_weighted_eval_ratio_FL_0.5["Precision.Pos Pred Value"],
    rf_weighted_eval_ratio_FL_1["Precision.Pos Pred Value"],
    rf_weighted_eval_ratio_FL_1.5["Precision.Pos Pred Value"],
    rf_weighted_eval_ratio_FL_2["Precision.Pos Pred Value"],
    rf_weighted_eval_ratio_FL_3["Precision.Pos Pred Value"],
    rf_weighted_eval_ratio_FL_4["Precision.Pos Pred Value"]
  ),
  F1_Score = c(
    rf_weighted_eval_ratio_0.5["F1_Score.F1"],
    rf_weighted_eval_ratio_1["F1_Score.F1"],
    rf_weighted_eval_ratio_1.5["F1_Score.F1"],
    rf_weighted_eval_ratio_2["F1_Score.F1"],
    rf_weighted_eval_ratio_3["F1_Score.F1"],
    rf_weighted_eval_weight_inverse["F1_Score.F1"],
    rf_weighted_eval_ratio_WCE_0.5["F1_Score.F1"],
    rf_weighted_eval_ratio_WCE_1["F1_Score.F1"],
    rf_weighted_eval_ratio_WCE_1.5["F1_Score.F1"],
    rf_weighted_eval_ratio_WCE_2["F1_Score.F1"],
    rf_weighted_eval_ratio_WCE_3["F1_Score.F1"],
    rf_weighted_eval_ratio_WCE_4["F1_Score.F1"],
    rf_weighted_eval_ratio_FL_0.5["F1_Score.F1"],
    rf_weighted_eval_ratio_FL_1["F1_Score.F1"],
    rf_weighted_eval_ratio_FL_1.5["F1_Score.F1"],
    rf_weighted_eval_ratio_FL_2["F1_Score.F1"],
    rf_weighted_eval_ratio_FL_3["F1_Score.F1"],
    rf_weighted_eval_ratio_FL_4["F1_Score.F1"]
  )
)

best_f1_index <- which.max(rf_weighted_eval_df$F1_Score)

# Generate the table with row highlight
kable(rf_weighted_eval_df, digits = 4, caption = "Performance Across Different Weighting Schema") %>%
  kable_styling(bootstrap_options = c("striped", "hover", "condensed")) %>%
  row_spec(best_f1_index, background = "#90EE90")



##### 3.6.6 RF + WCE + SMOTE

After evaluating multiple class imbalance handling strategies, it is now clear that Weighted Cross-Entropy (WCE) performs best in this context. WCE adjusts the loss function by assigning more importance to the minority class, helping the model focus on harder-to-classify examples and address class imbalance effectively.

Building on this, the next step is to combine WCE with sampling techniques to further enhance performance. Specifically, we will integrate WCE with:

SMOTE (Synthetic Minority Oversampling Technique): which generates synthetic samples for the minority class to balance the dataset.

SMOTE-ENN (SMOTE + Edited Nearest Neighbors): which combines oversampling with data cleaning by removing ambiguous and noisy samples using nearest neighbors.

This hybrid approach aims to leverage both algorithm-level (WCE) and data-level (SMOTE/SMOTE-ENN) methods to improve predictive accuracy—especially for the minority class.

We will implement and evaluate the following combinations:

Weighted Cross-Entropy (weight = 1) + SMOTE

Weighted Cross-Entropy (weight = 1) + SMOTE-ENN

The models will be compared using performance metrics such as F1-score, precision, recall, and AUC on validation and test datasets.


In [ ]:

w <- 1

model_weights_WCE <- c(
  negative = - w * log(p_negative),
  positive = - log(1 - p_negative)  # log(1 - p_major)
)



weighted_rf_WCE_smote <- train(Class ~ .,
                     data = data_train_smote,
                     method = "ranger",
                     trControl = train_control,
                     metric = "ROC",
                     class.weights = model_weights_WCE,
                     num.trees = 100, ## Default is 500
                     importance = 'permutation'
                    )


weighted_rf_eval_WCE_smote <- evaluate_model(weighted_rf_WCE_smote, 
               data_valid_smote,
               data_test_smote)

cm_weighted_rf_WCE_smote  <- weighted_rf_eval_WCE_smote$cm

rf_weighted_eval_WCE_smote <- eval_fnc(cm_weighted_rf_WCE_smote)


##### 3.6.7 RF + FL + SMOTE-ENN

In [ ]:

weighted_rf_WCE_smote_enn <- train(Class ~ .,
                     data = data_train_smote_enn,
                     method = "ranger",
                     trControl = train_control,
                     metric = "ROC",
                     class.weights = model_weights_WCE,
                     num.trees = 100, ## Default is 500
                     importance = 'permutation'
                    )


weighted_rf_eval_WCE_smote_enn <- evaluate_model(weighted_rf_WCE_smote_enn, 
               data_valid_smote_enn ,
               data_test_smote_enn)

cm_weighted_rf_WCE  <- weighted_rf_eval_WCE_smote_enn$cm

rf_weighted_eval_WCE_smote_enn <- eval_fnc(cm_weighted_rf_WCE)


### 3.7 Quantification 

Since the evaluation score was low, I am now using quantification methods using two approaches for our best model so far, weighted random forest with different weights, and our baseline model. 

*Probabilistic Classify & Count (PCC): adding the predicted probabilities.

*Adjusted Classify & Count (ACC): Using classifier's known error rates (calculated from a training set) to correct the raw prediction counts from a test set.

#### 3.7.1 PCC (Weighted RF)

In [ ]:

ratio_to_try <- c(0.5, 1.0, 1.5, 2.0, 3.0) # Example ratios relative to the inverse frequency
boots <- seq(1:100)

R <- length(ratio_to_try)
B <- length(boots)

# PCC <- matrix(NA_real_, nrow = R, ncol = B,
#                     dimnames = list(r = 1:R, b = 1:B))
# 
# for (bb in seq_len(B)){
#   print(bb)
# for (rr in seq_len(R)){
#   r <- ratio_to_try[[rr]]
#   model_weights <- c(
#     negative = (n_negative_train + n_positive_train) / (r * n_negative_train),
#     positive =  (n_negative_train + n_positive_train) / (r * n_positive_train)
#   )
# 
# weighted_rf <- train(Class ~ .,
#                      data = data_train,
#                      method = "ranger",
#                      trControl = train_control,
#                      metric = "ROC",
#                      class.weights = model_weights,
#                      num.trees = 100, ## Default is 500
#                      importance = 'permutation'
#                     )
# 
# PCC[rr,bb] <- round(sum(predict(weighted_rf, 
#         newdata = data_test, type = "prob")[, "positive"]))
# }
# }
actual_count <- sum(data_test$Class == 'positive')
# 
# saveRDS(PCC, 'PCC.rds')

PCC <- read_rds('PCC.rds')


# Make sure row names reflect the actual ratios you used
rownames(PCC) <- as.character(ratio_to_try)
colnames(PCC) <- paste0("boot_", boots)

# Long table of all results
df_pcc <- data.frame(
  ratio = rep(ratio_to_try, times = ncol(PCC)),
  boot  = rep(boots, each = nrow(PCC)),
  PCC   = as.vector(PCC)
)

# Summaries per ratio (percentile bootstrap CI)
sum_pcc <- df_pcc %>%
  group_by(ratio) %>%
  summarize(
    median = median(PCC, na.rm = TRUE),
    lwr    = quantile(PCC, 0.025, na.rm = TRUE),
    upr    = quantile(PCC, 0.975, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(`95% CI` = sprintf("[%.0f, %.0f]", lwr, upr)) %>%
  select(ratio, median, `95% CI`) %>%
  arrange(ratio)


n_total_train <- n_negative_train + n_positive_train

p_positive <- n_positive_train / n_total_train
p_negative <- n_negative_train / n_total_train

wce <- c(0.5, 1, 1.5, 2, 3, 4)
W <- length(wce)

# PCC_wce <- matrix(NA_real_, nrow = W, ncol = B,
#                     dimnames = list(w = 1:W, b = 1:B))
# 
# for (bb in seq_len(B)){
#   print(bb)
# for (ww in seq_len(W)){
# 
# w <- wce[[ww]]
# 
# model_weights_WCE <- c(
#   negative = - w * log(p_negative),
#   positive = - log(1 - p_negative)  # log(1 - p_major)
# )
# 
# weighted_rf_WCE <- train(Class ~ .,
#                      data = data_train,
#                      method = "ranger",
#                      trControl = train_control,
#                      metric = "ROC",
#                      class.weights = model_weights_WCE,
#                      num.trees = 100, ## Default is 500
#                      importance = 'permutation'
#                     )
# 
# PCC_wce[ww,bb] <- round(sum(predict(weighted_rf_WCE,
#         newdata = data_test, type = "prob")[, "positive"]))
# }}
# 
# saveRDS(PCC_wce, 'PCC_wce.rds')
PCC_wce <- read_rds('PCC_wce.rds')

rownames(PCC_wce) <- as.character(wce)
colnames(PCC_wce) <- paste0("boot_", boots)

# Long table of all results
df_pcc <- data.frame(
  weights = rep(wce, times = ncol(PCC_wce)),
  boot  = rep(boots, each = nrow(PCC_wce)),
  PCC_wce   = as.vector(PCC_wce)
)

# Summaries per ratio (percentile bootstrap CI)
sum_pcc_wce <- df_pcc %>%
  group_by(weights) %>%
  summarize(
    median = median(PCC_wce, na.rm = TRUE),
    lwr    = quantile(PCC_wce, 0.025, na.rm = TRUE),
    upr    = quantile(PCC_wce, 0.975, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(`95% CI` = sprintf("[%.0f, %.0f]", lwr, upr)) %>%
  select(weights, median, `95% CI`) %>%
  arrange(weights)


sum_pcc %>%
  kbl(
    digits = 3,
    align = "c",
    col.names = c("Weight ratio r", "Median PCC", "95% CI (bootstrap)")
  ) %>%
  kable_paper(full_width = FALSE) %>%
  add_header_above(c(" " = 1, "PCC summary across bootstraps - Adjusting R" = 2))


sum_pcc_wce %>%
  kbl(
    digits = 3,
    align = "c",
    col.names = c("Weight ratio w", "Median PCC", "95% CI (bootstrap)")
  ) %>%
  kable_paper(full_width = FALSE) %>%
  add_header_above(c(" " = 1, "PCC summary across bootstraps - WCE Adjusting W" = 2))



Actual Prevalence in Test Set is `r actual_count`.

#### 3.7.2 ACC (Weighted RF)

In [ ]:
gc()
ratio_to_try <- c(0.5, 1.0, 1.5, 2.0, 3.0) # Example ratios relative to the inverse frequency
boots <- seq(1:100)

R <- length(ratio_to_try)
B <- length(boots)

# ACC <- matrix(NA_real_, nrow = R, ncol = B,
#                     dimnames = list(r = 1:R, b = 1:B))

# for (bb in seq_len(B)){
#   print(bb)
# for (rr in seq_len(R)){
#   r <- ratio_to_try[[rr]]
#   model_weights <- c(
#     negative = (n_negative_train + n_positive_train) / (r * n_negative_train),
#     positive =  (n_negative_train + n_positive_train) / (r * n_positive_train)
#   )
# 
# weighted_rf <- train(Class ~ .,
#                      data = data_train,
#                      method = "ranger",
#                      trControl = train_control,
#                      metric = "ROC",
#                      class.weights = model_weights,
#                      num.trees = 100, ## Default is 500
#                      importance = 'permutation'
#                     )
# 
# 
# cm_weighted_rf_training_tbl <- evaluate_model(weighted_rf,
#                data_valid,
#                data_train)$cm$table
# 
# TP <- cm_weighted_rf_training_tbl[2, 2] # Predicted: Positive, Actual: Positive
# FP <- cm_weighted_rf_training_tbl[2, 1] # Predicted: Positive, Actual: Negative
# TN <- cm_weighted_rf_training_tbl[1, 1] # Predicted: Negative, Actual: Negative
# FN <- cm_weighted_rf_training_tbl[1, 2] # Predicted: Negative, Actual: Positive
# 
# tpr <- TP / (TP + FN)
# fpr <- FP / (FP + TN)
# 
# ACC[rr,bb] <- round(
#   (sum(predict(weighted_rf_default,
#         newdata = data_test, type = "prob")[, "positive"]) - fpr) / (tpr - fpr)
#   )
# }
# }
# saveRDS(ACC, 'ACC.rds')
# saveRDS(ACC_extra, 'ACC_extra.rds')

actual_count <- sum(data_test$Class == 'positive')

ACC <- read_rds('ACC.rds')
rownames(ACC) <- as.character(ratio_to_try)
colnames(ACC) <- paste0("boot_", boots)

# Long table of all results
df_ACC <- data.frame(
  ratio = rep(ratio_to_try, times = ncol(ACC)),
  boot  = rep(boots, each = nrow(ACC)),
  ACC   = as.vector(ACC)
)


# Summaries per ratio (percentile bootstrap CI)
sum_ACC <- rbind(df_ACC) %>%
  drop_na()%>%
  group_by(ratio) %>%
  summarize(
    median = median(ACC, na.rm = TRUE),
    lwr    = quantile(ACC, 0.025, na.rm = TRUE),
    upr    = quantile(ACC, 0.975, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(`95% CI` = sprintf("[%.0f, %.0f]", lwr, upr)) %>%
  select(ratio, median, `95% CI`) %>%
  arrange(ratio)


# ACC_wce <- matrix(NA_real_, nrow = W, ncol = B,
#                     dimnames = list(w = 1:W, b = 1:B))
# 
# for (bb in seq_len(B)){
#   print(bb)
# for (ww in seq_len(W)){
# 
# w <- wce[[ww]]
# 
# model_weights_WCE <- c(
#   negative = - w * log(p_negative),
#   positive = - log(1 - p_negative)  # log(1 - p_major)
# )
# 
# weighted_rf_WCE <- train(Class ~ .,
#                      data = data_train,
#                      method = "ranger",
#                      trControl = train_control,
#                      metric = "ROC",
#                      class.weights = model_weights_WCE,
#                      num.trees = 100, ## Default is 500
#                      importance = 'permutation'
#                     )
# 
# cm_weighted_rf_training_tbl <- evaluate_model(weighted_rf_WCE,
#                data_valid,
#                data_train)$cm$table
# 
# TP <- cm_weighted_rf_training_tbl[2, 2] # Predicted: Positive, Actual: Positive
# FP <- cm_weighted_rf_training_tbl[2, 1] # Predicted: Positive, Actual: Negative
# TN <- cm_weighted_rf_training_tbl[1, 1] # Predicted: Negative, Actual: Negative
# FN <- cm_weighted_rf_training_tbl[1, 2] # Predicted: Negative, Actual: Positive
# 
# tpr <- TP / (TP + FN)
# fpr <- FP / (FP + TN)
# 
# ACC_wce[ww,bb] <- round(
#   (sum(predict(weighted_rf_WCE,
#         newdata = data_test, type = "prob")[, "positive"]) - fpr) / (tpr - fpr)
#   )
# 
# }}

# saveRDS(ACC_wce, 'ACC_wce.rds')

# Pretty table with kableExtra
# sum_ACC %>%
#   kbl(
#     digits = 3,
#     align = "c",
#     col.names = c("Weight ratio r", "Median ACC", "95% CI (bootstrap)")
#   ) %>%
#   kable_paper(full_width = FALSE) %>%
#   add_header_above(c(" " = 1, "summary across bootstraps - Adjusting R" = 2))
# 

ACC_wce <- read_rds('ACC_wce.rds')

rownames(ACC_wce) <- as.character(wce)
colnames(ACC_wce) <- paste0("boot_", boots)

# Long table of all results
df_ACC <- data.frame(
  weights = rep(wce, times = ncol(ACC_wce)),
  boot  = rep(boots, each = nrow(ACC_wce)),
  ACC_wce   = as.vector(ACC_wce)
)

# Summaries per ratio (percentile bootstrap CI)
sum_ACC_wce <- df_ACC %>%
  group_by(weights) %>%
  summarize(
    median = median(ACC_wce, na.rm = TRUE),
    lwr    = quantile(ACC_wce, 0.025, na.rm = TRUE),
    upr    = quantile(ACC_wce, 0.975, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(`95% CI` = sprintf("[%.0f, %.0f]", lwr, upr)) %>%
  select(weights, median, `95% CI`) %>%
  arrange(weights)


sum_ACC %>%
  kbl(
    digits = 3,
    align = "c",
    col.names = c("Weight ratio r", "Median ACC", "95% CI (bootstrap)")
  ) %>%
  kable_paper(full_width = FALSE) %>%
  add_header_above(c(" " = 1, "ACC summary across bootstraps - Adjusting R" = 2))


sum_ACC_wce %>%
  kbl(
    digits = 3,
    align = "c",
    col.names = c("Weight ratio w", "Median ACC", "95% CI (bootstrap)")
  ) %>%
  kable_paper(full_width = FALSE) %>%
  add_header_above(c(" " = 1, "ACC summary across bootstraps - WCE Adjusting W" = 2))




Actual Prevalence in Test Set is `r actual_count`.

#### 3.7.3 PCC (Baseline GLM)

In [ ]:

PCC <- round(sum(predict(glm_baseline, 
        newdata = data_test, type = "prob")[, "positive"]))

actual_count <- sum(data_test$Class == 'positive')


Actual Prevalence in Test Set for is `r actual_count` and the predicted PCC from GLM Baseline is `r PCC`.

#### 3.7.4 ACC (Baseline GLM)

In [ ]:

cm_glm_training_tbl <- evaluate_model(glm_baseline, 
               data_valid,
               data_train)$cm$table

TP <- cm_glm_training_tbl[2, 2] # Predicted: Positive, Actual: Positive
FP <- cm_glm_training_tbl[2, 1] # Predicted: Positive, Actual: Negative
TN <- cm_glm_training_tbl[1, 1] # Predicted: Negative, Actual: Negative
FN <- cm_glm_training_tbl[1, 2] # Predicted: Negative, Actual: Positive

tpr <- TP / (TP + FN)
fpr <- FP / (FP + TN)

ACC <- round(
  (sum(predict(glm_baseline, 
        newdata = data_test, type = "prob")[, "positive"]) - fpr) / (tpr - fpr)
  )


Actual Prevalence in Test Set for is `r actual_count` and the predicted ACC from GLM Baseline is `r ACC`.